# 08 - Beta-beta comparison

Compares effect sizes variant by variant between two sets of summary
statistics, and reports correlation, regression slope and sign concordance.
Two comparisons are run here:

1. **survival vs cross-sectional** - the two HY3 analyses in this study. They
   share an effect scale, so the axes share one symmetric range and variants
   are coloured by where they reach genome-wide significance.
2. **PD risk vs HY3 progression** - an external risk GWAS against one
   progression analysis. The effect alleles have to be aligned, the scales are
   not comparable, and the variants come from two different hit lists, so the
   colouring is by nominating list instead.

Both inputs need the columns `SNP`, `BETA`, `SE` and `P`;
`rename_summary_stats()` maps the METAL/LocusZoom names onto them and coerces
the effect columns to float.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gwas import config as cfg
from gwas import betabeta as bb

pd.options.mode.chained_assignment = None

cfg.describe()

In [ ]:
# Mount the release and working buckets with gcsfuse (run once per VM).
for cmd in cfg.mount_commands():
    print(cmd)

# 1. Survival vs cross-sectional

## Variants to compare

In [ ]:
# Curated variant table: one row per reported variant, with the analysis it
# came from and the nearest gene.
curated = pd.read_csv(cfg.prep_path('meta', 'forest_plots', 'curated_variants.csv'))
display(curated)

EXCLUDE_GENES = []    # e.g. pseudogenes that are not worth plotting
target_snps = curated[curated.Variant.isin(['HY3_LR', 'HY3_CPH'])
                      & ~curated.Gene.isin(EXCLUDE_GENES)].dbSNP_ID.tolist()
print(f'{len(target_snps)} variants')

## Load the two analyses

These are the annotated METAL outputs written by `06_clean_results.ipynb`;
`rsid` becomes `SNP` so the two can be merged on a common identifier.

In [ ]:
def load_meta(analysis):
    path = (cfg.prep_path('meta', 'METAL', analysis, 'locuszoom', 'combined')
            / f'METAL_{analysis}.tsv.gz')
    df = pd.read_csv(path, compression='gzip', sep='\t')
    return bb.rename_summary_stats(df)


surv = load_meta('SURV')
cs_disc = load_meta('CS_disc')

print(sorted(set(surv.columns) & {'SNP', 'BETA', 'SE', 'P', 'A1', 'A2', 'EAF'}))
print(f'SURV: {len(surv):,} variants | CS_disc: {len(cs_disc):,} variants')

## Compare

Both analyses are on the same scale, so `axis_scale` stays at its default of
`'symmetric'`: shared limits centred on zero and an equal aspect, which makes
y = x the true diagonal.

`annotate_all` labels every plotted variant; with a long list use
`annotate_discordant=N` instead to label only the N variants whose effects
disagree most.

In [ ]:
out_dir = cfg.prep_path('meta', 'beta_beta')
out_dir.mkdir(parents=True, exist_ok=True)

fig, ax, metrics, merged = bb.compare_effect(
    surv, cs_disc,
    snplist=target_snps,
    label=['SURV', 'CS_disc', 'Both', 'None'],
    annotate_all=True,
    plot_error_bars=True,
    save=out_dir / 'SURV_vs_CS_disc_hits.png',
    return_data=True,
)
plt.show()
display(pd.Series(metrics))

## Genome-wide comparison

The same comparison over every shared variant rather than the reported hits.
Error bars and labels are off b/c of computational cost.

In [ ]:
fig, ax, metrics = bb.compare_effect(
    surv, cs_disc,
    label=['SURV', 'CS_disc', 'Both', 'None'],
    annotate_discordant=5,
    plot_error_bars=False,
    s=6,
    save=out_dir / 'SURV_vs_CS_disc_all.png',
)
plt.show()
display(pd.Series(metrics))

## Duplicated variants

`compare_effect` keeps the lowest-p row per SNP

In [ ]:
for df, name in ((surv, 'SURV'), (cs_disc, 'CS_disc')):
    bb.report_duplicates(df[df.SNP.isin(target_snps)], name)

# 2. PD risk vs HY3 progression

Comparing the progression GWAS against an external PD **risk** GWAS, to see
whether the variants that make PD more likely are also the ones that make it
progress faster. Three things differ from part 1:

* the two GWAS report effects against their own ALT allele, so the risk betas
  are put on the progression effect allele with `align_alleles()` before
  anything is compared;
* a known risk locus carries a far larger log OR than anything in progression,
  so `axis_scale='fit'` pads each axis around its own effects (keeping zero in
  view) rather than forcing one shared range that would flatten the y axis;
* the variants come from two separate hit lists, so `nominated_by=` colours
  each point by the list that nominated it rather than by p-value.

In [ ]:
ANALYSIS = 'CS_disc'    # progression analysis to compare against risk

risk_dir = cfg.prep_path('meta', 'risk')

progression = load_meta(ANALYSIS)
risk = bb.rename_summary_stats(
    pd.read_csv(risk_dir / 'GP2_EUR_GWAS.gz', compression='gzip', sep='\t'))

print(f'risk: {len(risk):,} variants | {ANALYSIS}: {len(progression):,} variants')

## Variants to compare

The union of the two hit lists. The risk hits arrive as `chr:pos:ref:alt`
IDs, so both lists are translated to rsIDs through the progression table -
that is the identifier the two GWAS are merged on.

In [ ]:
# Variants reported by the risk GWAS, as chr:pos:ref:alt IDs.
risk_hits = pd.read_csv(risk_dir / 'risk_variants.txt', sep='\t')['SNP'].tolist()

# The progression hits from the curated table, as the same kind of ID.
prog_hits = progression.loc[progression.SNP.isin(target_snps), 'MarkerName'].tolist()


def to_rsids(marker_names):
    return progression.loc[progression.MarkerName.isin(marker_names), 'SNP'].tolist()


risk_rsids = to_rsids(risk_hits)
prog_rsids = to_rsids(prog_hits)
compare_snps = list(dict.fromkeys(risk_rsids + prog_rsids))
print(f'{len(risk_rsids)} risk-nominated + {len(prog_rsids)} progression hits '
      f'-> {len(compare_snps)} variants to compare')

## Align and compare

`align_alleles()` flips the risk betas (and allele frequencies) onto the
progression effect allele and drops any variant whose alleles cannot be
reconciled. Without it a concordant variant reported against the other allele
would land in the wrong quadrant.

In [ ]:
prog_df = bb.load_and_prep(progression, 'Progression', snplist=compare_snps)
risk_df = bb.load_and_prep(risk, 'Risk', snplist=compare_snps)
risk_df = bb.align_alleles(risk_df, prog_df, name='Risk')

# Label the risk variants that come out most significant in progression.
top_risk = prog_df[prog_df.SNP.isin(risk_rsids)].nsmallest(5, 'P')
display(top_risk[['SNP', 'BETA', 'SE', 'P']])

In [ ]:
Y_LABELS = {'CS_disc': 'Baseline HY3', 'SURV': 'Survival to HY3'}

fig, ax, metrics = bb.compare_effect(
    risk_df, prog_df,
    label=['Risk', 'Progression', 'Both', 'None'],
    nominated_by={'Risk': risk_rsids, 'Progression': prog_rsids},
    annotate_snps=top_risk.SNP.tolist(),
    annotate_discordant=0,
    axis_scale='fit',
    xlabel='PD risk',
    ylabel=Y_LABELS[ANALYSIS],
    save=out_dir / f'risk_vs_{ANALYSIS}.png',
)
plt.show()
display(pd.Series(metrics))